# Projet Pokémon — recommandation de gagnant en duel

Ce notebook présente le pipeline d’intelligence artificielle de **Pokémon Duel MLOps**, dans le même esprit que `Donné/note.ipynb` : comprendre les données, préparer les duels, entraîner plusieurs modèles, les comparer et tester une recommandation.

**Objectif :** à partir de deux Pokémon, estimer lequel a le plus de chances de gagner selon le Pokédex, les types et les combats fournis. Le modèle apprend à partir des 50 000 duels de `Donné/combats.csv`.

> La prédiction est une estimation statistique. Elle ne simule pas chaque attaque, objet ou stratégie du jeu.

# ML/DL - M2 GID ENI :
1. JAONARIEAFY Josue Silvano 3818
2. RAZAFIARISON Eraldo Albert 3815
3. JOEL Stéphen 3834
4. RAKOTONDRANAIVO Fahazavantsoa Harikasaina 3877
5. RASOAMANAMPY Mbolatiana Victoire 3879
6. RANDRIAMANANTENA Mahatradraibe Jean Louis 3878


## 1. Importer les bibliothèques et les fonctions du projet

On réutilise `ml/data.py` et `ml/train.py` afin que le notebook suive le même traitement que l’API déployée.

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

working_dir = Path.cwd().resolve()
project_candidates = [working_dir, *working_dir.parents]
PROJECT_ROOT = next(
    path for path in project_candidates
    if (path / "Donné" / "pokedex.csv").exists()
    and (path / "ml" / "train.py").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "ml"))

from data import (
    ARTIFACTS_DIR,
    FEATURES,
    build_battle_dataset,
    build_pair_features,
    load_data,
    symmetric_first_probability,
)
from train import train

print(f"Dossier du projet : {PROJECT_ROOT}")
print(f"Dossier des artefacts : {ARTIFACTS_DIR}")

ModuleNotFoundError: No module named 'matplotlib'

## 2. Charger les sources de données

- `pokedex.csv` décrit les Pokémon (statistiques, types et statut légendaire).
- `combats.csv` contient les deux participants et l’identifiant du gagnant.
- `poke_type_chart.csv` donne l’efficacité des types.

Le chargement applique aussi les contrôles de colonnes, d’identifiants et de gagnants de l’application.

In [ ]:
pokedex, combats, type_chart = load_data()

print(f"Pokémon : {pokedex.shape[0]} lignes, {pokedex.shape[1]} colonnes")
display(pokedex.head())

print(f"\nCombats : {combats.shape[0]} lignes, {combats.shape[1]} colonnes")
display(combats.head())

print(f"\nTable des types : {type_chart.shape[0]} types")
display(type_chart.iloc[:5, :5])

## 3. Explorer et préparer les données

Les identifiants sont vérifiés lors du chargement. Les types absents sont représentés par une chaîne vide, les noms inconnus sont complétés à partir du numéro et la colonne légendaire est convertie en 0/1.

In [ ]:
print("Valeurs manquantes dans le Pokédex :")
display(pokedex.isna().sum().to_frame("valeurs_manquantes").T)

print("\nRépartition des gagnants en première ou seconde position :")
display(
    combats["GAGNANT"]
    .eq(combats["POKEMON_PREMIER"])
    .map({True: "Premier Pokémon", False: "Second Pokémon"})
    .value_counts()
    .to_frame("nombre de combats")
)

print("\nPokémon légendaires :")
display(pokedex["LEGENDAIRE"].value_counts().rename(index={0: "Non", 1: "Oui"}).to_frame("nombre"))

## 4. Construire les caractéristiques des duels

Pour chaque duel, les entrées sont les différences entre le premier et le second Pokémon : points de vie, attaques, défenses, vitesse et statut légendaire. On ajoute aussi l’efficacité des types dans chaque sens.

La cible vaut `1` si le premier Pokémon a gagné et `0` si c’est le second. Ces différences rendent la représentation interprétable dans les deux sens.

In [ ]:
X, y = build_battle_dataset(combats, pokedex, type_chart)

print(f"Matrice des caractéristiques : {X.shape}")
print(f"Nombre de caractéristiques : {len(FEATURES)}")
display(X.head())

print("\nCible : 1 = premier Pokémon gagnant, 0 = second Pokémon gagnant")
display(y.value_counts().rename(index={0: "Second gagnant", 1: "Premier gagnant"}).to_frame("combats"))

## 5. Séparer les données et comparer les modèles

La séparation apprentissage/test est faite **avant** de retourner l’ordre des adversaires : les deux versions d’un même duel ne peuvent donc pas se retrouver des deux côtés de la séparation. Le pipeline du projet complète ensuite les duels d’apprentissage par leur version inversée et compare une régression logistique, une Random Forest et un réseau de neurones MLP.

L’accuracy et le F1 macro sont calculés sur le jeu de test. Le modèle retenu et les graphiques sont enregistrés dans `ml/artifacts/`.

In [ ]:
metrics = train()

resultats = pd.DataFrame(metrics["classification"]["resultats"]).T
resultats.index.name = "modèle"
resultats = resultats.sort_values("f1_macro", ascending=False)

print(f"Modèle retenu : {metrics['classification']['modele_retenu']}")
print(
    f"Jeu d'entraînement : {metrics['n_combats_entrainement']:,} combats | "
    f"Jeu de test : {metrics['n_combats_test']:,} combats"
)
display(resultats.style.format({"accuracy": "{:.2%}", "f1_macro": "{:.2%}"}))

## 6. Examiner les graphiques d’évaluation

La matrice de confusion montre les classes prédites et réelles. L’importance des caractéristiques vient de la Random Forest. La courbe d’apprentissage illustre l’évolution des scores à mesure que le modèle voit davantage de données.

In [ ]:
for chart_name in ("confusion_matrix", "feature_importance", "learning_curve"):
    chart_path = ARTIFACTS_DIR / "charts" / f"{chart_name}.png"
    print(chart_name.replace("_", " ").title())
    display(Image(filename=str(chart_path)))

## 7. Essayer une prédiction et sauvegarder le modèle

Exemple : comparer Bulbasaur (nº 1) à Mega Venusaur (nº 4). La probabilité est symétrisée entre les deux ordres, comme dans l’API. Le fichier du modèle, les métriques et les graphiques créés ici sont ceux réutilisés par l’application.

In [ ]:
import joblib

premier_id, second_id = 1, 4
pokemon_par_id = pokedex.set_index("NUMERO")
duel_features = build_pair_features(
    premier_id,
    second_id,
    pokedex,
    type_chart,
)
duel_X = pd.DataFrame([duel_features], columns=FEATURES)
bundle = joblib.load(ARTIFACTS_DIR / "models.joblib")
modele = bundle["classifier"]
probabilite_premier = symmetric_first_probability(modele, duel_X)[0]

nom_premier = pokemon_par_id.loc[premier_id, "NOM"]
nom_second = pokemon_par_id.loc[second_id, "NOM"]
recommande = nom_premier if probabilite_premier >= 0.5 else nom_second
print(f"{nom_premier} : {probabilite_premier:.1%}")
print(f"{nom_second} : {1 - probabilite_premier:.1%}")
print(f"Recommandation du modèle : {recommande}")

print(f"\nModèle sauvegardé : {ARTIFACTS_DIR / 'models.joblib'}")
print(f"Métriques sauvegardées : {ARTIFACTS_DIR / 'metrics.json'}")

## 8. Déploiement et utilisation de l’application

Après l’entraînement, depuis un terminal ouvert à la racine du projet :

```powershell
uvicorn app:app --reload --host 127.0.0.1 --port 8000
```

- Interface web : <http://127.0.0.1:8000>
- Documentation interactive de l’API : <http://127.0.0.1:8000/docs>
- Santé du service : <http://127.0.0.1:8000/api/health>

Pour les tests du projet, exécutez `python -m pytest -v`. Le déploiement Docker/Render est décrit dans le `README.md` du projet. Les probabilités restent des estimations apprises sur les données fournies et ne garantissent pas le résultat d’un combat.